In [37]:
import pandas as pd
import numpy as np

In [38]:
# Import file and clean headers and footers to analyze properly
hi_forecast = pd.read_csv(
    '../data/raw/hawaii_employment_forecast_raw.csv',
    engine     = "python",
    skiprows   = 4,
    skipfooter = 8,
    header     = None,
    names    = [
        'SOC_Code', 'Occupation_Title',
        'Employment_2024', 'Employment_2034',
        'Growth_Net', 'Growth_Pct',
        'Annual_Growth',
        'Openings_Change', 'Openings_Transfer', 'Openings_Exits', 'Openings_Total',
        'Edu_Education', 'Edu_Work_Exp', 'Edu_Job_Training',
        'Median_Wage', 'SOC_Level'
    ],
    keep_default_na = False,  # stops pandas treating 'None' as NaN
    na_values  = ['*', '**', '']   # only treat these as NaN
)

In [39]:
# Variable Columns to change into numerical data types 
number_cols = ['Employment_2024', 'Employment_2034',
               'Growth_Net', 'Growth_Pct',
               'Annual_Growth',
               'Openings_Change', 'Openings_Transfer', 'Openings_Exits', 'Openings_Total',
               'Median_Wage', 'SOC_Level']

In [40]:
# Function to clean columns into numerics for data analyses
def clean_numeric_col(col):
    # 1. Force to string but preserve NaN
    col_str = col.astype(str).str.strip()
    
    # 2. Check for percentage signs BEFORE removing them
    is_percentage = col_str.str.contains('%', na=False).any()
    
    # 3. Clean non-numeric characters
    cleaned = col_str.str.replace(r'[$,% ]', '', regex=True).str.strip()
    
    # 4. Handle empty strings
    cleaned = cleaned.replace(['', 'nan', 'None'], np.nan)
    
    # 5. Convert to numeric
    numeric_col = pd.to_numeric(cleaned, errors='coerce')
    
    # 6. Divide percentages by 100
    if is_percentage:
        numeric_col = numeric_col / 100
        
    return numeric_col

In [41]:
hi_forecast[number_cols] = hi_forecast[number_cols].apply(clean_numeric_col)

In [42]:
# Highest Education Degree Sorted for Categorical Usage 
Edu_Education_order = [
    'None', 'HS/equiv', 'Some college', 'Postsec',
    "Associate's", "Bachelor's", "Master's", 'Doct/Prof'
]
hi_forecast['Edu_Education'] = pd.Categorical(
    hi_forecast['Edu_Education'],
    categories = Edu_Education_order,
    ordered    = True
)

hi_forecast['Edu_Education'] = hi_forecast['Edu_Education'].astype('category')

# Work Experience Needed Sorted for Categorical Usage 
Edu_Work_Exp_order = ['None', '<5 yrs', '5+ yrs']
hi_forecast['Edu_Work_Exp'] = pd.Categorical(
    hi_forecast['Edu_Work_Exp'],
    categories = Edu_Work_Exp_order,
    ordered    = True
)

hi_forecast['Edu_Work_Exp'] = hi_forecast['Edu_Work_Exp'].astype('category')

Edu_Training_order = [
    "None",          # 0
    "Short OJT",     # 1
    "Moderate OJT",  # 2
    "Long OJT",      # 3
    "Apprentice",    # 4
    "Intern/Resident" # 5
]

hi_forecast['Edu_Job_Training'] = pd.Categorical(
    hi_forecast['Edu_Job_Training'],
    categories = Edu_Training_order,
    ordered    = True
)

hi_forecast['Edu_Job_Training'] = hi_forecast['Edu_Job_Training'].astype('category')

# Standard Occupation Code (SOC) Hierachy - (1 - ALL OCCUPATIONS, 2 - MAJOR GROUP, 4 - SPECIFIC OCCUPATION)
# First 2 digits:     Denotes Major Group
# 3rd digit:          Denotes Minor Group
# 4th and 5th digits: Denotes Broad Occupation
# 6th digit:          Denotes Detailed Occupation
hi_forecast['SOC_Level'] = hi_forecast['SOC_Level'].astype('category')

In [43]:
# Verify format looks right
hi_forecast.head(10)

,SOC_Code,Occupation_Title,Employment_2024,Employment_2034,Growth_Net,Growth_Pct,Annual_Growth,Openings_Change,Openings_Transfer,Openings_Exits,Openings_Total,Edu_Education,Edu_Work_Exp,Edu_Job_Training,Median_Wage,SOC_Level
0,11-0000,Management Occupations,53630,56640,3020,0.056,0.005,300.0,2600.0,1600.0,4510.0,NaN,NaN,NaN,116040.0,2
1,11-1011,Chief Executives,820,850,20,0.029,0.003,NaN,30.0,30.0,60.0,Bachelor's,5+ yrs,None,281640.0,4
2,11-1021,General and Operations Managers,13360,13940,570,0.043,0.004,60.0,740.0,320.0,1110.0,Bachelor's,5+ yrs,None,108180.0,4
3,11-2021,Marketing Managers,540,560,30,0.047,0.005,NaN,30.0,10.0,40.0,Bachelor's,5+ yrs,None,117770.0,4
4,11-2022,Sales Managers,1850,1910,70,0.036,0.004,10.0,90.0,40.0,140.0,Bachelor's,<5 yrs,None,116340.0,4
5,11-2032,Public Relations Managers,350,370,10,0.040,0.004,NaN,20.0,10.0,30.0,Bachelor's,5+ yrs,None,105010.0,4
6,11-2033,Fundraising Managers,190,200,20,0.086,0.008,NaN,10.0,NaN,20.0,Bachelor's,5+ yrs,None,93910.0,4
7,11-3012,Administrative Services Managers,1840,1940,90,0.051,0.005,10.0,90.0,60.0,160.0,Bachelor's,<5 yrs,None,110840.0,4
8,11-3013,Facilities Managers,810,860,50,0.067,0.006,10.0,40.0,30.0,70.0,Bachelor's,<5 yrs,None,108750.0,4
9,11-3021,Computer and Information Systems Managers,1430,1610,180,0.124,0.012,20.0,70.0,30.0,110.0,Bachelor's,5+ yrs,None,141640.0,4


In [44]:
# Verify format looks right
hi_forecast.tail(10)

,SOC_Code,Occupation_Title,Employment_2024,Employment_2034,Growth_Net,Growth_Pct,Annual_Growth,Openings_Change,Openings_Transfer,Openings_Exits,Openings_Total,Edu_Education,Edu_Work_Exp,Edu_Job_Training,Median_Wage,SOC_Level
541,53-6099,"Transportation Workers, All Other",160,160,10,0.038,0.004,NaN,10.0,10.0,20.0,HS/equiv,None,Short OJT,59500.0,4
542,53-7021,Crane and Tower Operators,230,230,0,0.009,0.001,0.0,10.0,10.0,20.0,HS/equiv,<5 yrs,Moderate OJT,124260.0,4
543,53-7041,Hoist and Winch Operators,120,120,0,0.026,0.003,0.0,10.0,10.0,10.0,None,None,Short OJT,107100.0,4
544,53-7051,Industrial Truck and Tractor Operators,900,960,60,0.066,0.006,10.0,60.0,30.0,90.0,None,None,Short OJT,53640.0,4
545,53-7061,Cleaners of Vehicles and Equipment,2160,2240,80,0.036,0.004,10.0,160.0,120.0,300.0,None,None,Short OJT,37560.0,4
546,53-7062,"Laborers and Freight, Stock, and Material Move...",9470,10000,530,0.056,0.005,50.0,740.0,490.0,1280.0,None,None,Short OJT,44120.0,4
547,53-7064,"Packers and Packagers, Hand",2330,2200,-130,-0.055,-0.006,-10.0,170.0,130.0,290.0,None,None,Short OJT,35930.0,4
548,53-7065,Stockers and Order Fillers,8710,9780,1070,0.123,0.012,110.0,830.0,620.0,1550.0,None,None,Short OJT,39300.0,4
549,53-7081,Refuse and Recyclable Material Collectors,790,820,40,0.046,0.004,NaN,60.0,30.0,90.0,None,None,Short OJT,56270.0,4
550,53-7199,"Material Moving Workers, All Other",40,40,0,0.050,0.005,0.0,NaN,NaN,10.0,None,None,Short OJT,NaN,4


In [45]:
occupation_titles = [
'Reinforcing Iron and Rebar Workers',                 
'Elevator and Escalator Installers and Repairers',
'Layout Workers, Metal and Plastic',               
'Insulation Workers, Floor, Ceiling, and Wall',          
'Paving, Surfacing, and Tamping Equipment Operators',     
'Construction and Related Workers, All Other',            
'Millwrights',                                            
'Structural Metal Fabricators and Fitters',               
'Home Health Aides',                                      
'Stonemasons'
]

hi_forecast[hi_forecast['Occupation_Title'].isin(occupation_titles)]['Median_Wage']

Series([], Name: Median_Wage, dtype: float64)

In [46]:
SOC_things = [
'47-2171',
'47-4021',
'51-4192',
'47-2131',
'47-4099',
'47-2071',
'49-9044',
'51-2041',
'31-1121',
'47-2022'
]

hi_forecast[hi_forecast['SOC_Code'].isin(SOC_things)]

,SOC_Code,Occupation_Title,Employment_2024,Employment_2034,Growth_Net,Growth_Pct,Annual_Growth,Openings_Change,Openings_Transfer,Openings_Exits,Openings_Total,Edu_Education,Edu_Work_Exp,Edu_Job_Training,Median_Wage,SOC_Level


In [47]:
# Vertify data types are correct
hi_forecast.dtypes

SOC_Code               object
Occupation_Title       object
Employment_2024         int64
Employment_2034         int64
Growth_Net              int64
Growth_Pct            float64
Annual_Growth         float64
Openings_Change       float64
Openings_Transfer     float64
Openings_Exits        float64
Openings_Total        float64
Edu_Education        category
Edu_Work_Exp         category
Edu_Job_Training     category
Median_Wage           float64
SOC_Level            category
dtype: object

In [48]:
assert hi_forecast['Growth_Pct'].max() < 1.0, "Percentages should be decimals"
assert hi_forecast['SOC_Code'].notna().all(), "SOC codes should never be null"

In [50]:
# Export data-cleaned verison of hawaii_employment_forecast
hi_forecast.to_parquet('../data/processed/hawaii_employment_forecast_clean.parquet', index=False)